# FOXF1_bead — 03_live_fixed_alignment

**Feeds:** Fig 2c, ED Fig 3d

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 03 Live-Fixed Alignment

This notebook aligns the `2026-01-22` fixed small/large images to the already-registered live day2 coordinate system.

Outputs written here:
- `results/annotations/fixed_small_to_large_pair_transforms.tsv`
- `results/qc/03_live_fixed_alignment/fixed_small_large_alignment/*.png`
- `results/qc/03_live_fixed_alignment/fixed_small_large_alignment_summary.tsv`
- `results/annotations/live_fixed_alignment_transforms.tsv`
- `results/annotations/well_centroids_fixed_small_mapped.tsv`
- `results/qc/03_live_fixed_alignment/live_fixed_alignment/*.png`
- `results/qc/03_live_fixed_alignment/live_fixed_alignment_summary.tsv`

Run `01_small_large_alignment.ipynb` first.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from scripts import manual_well_annotation as mwa
from scripts import live_fixed_alignment as lfa
from scripts import live_fixed_quantification as lfq


## Paths

In [ ]:
LIVE_SMALL_LARGE_TSV = ROOT / "results/annotations/small_to_large_pair_transforms.tsv"
FIXED_SMALL_LARGE_TSV = ROOT / "results/annotations/fixed_small_to_large_pair_transforms.tsv"
LIVE_FIXED_TSV = ROOT / "results/annotations/live_fixed_alignment_transforms.tsv"
FIXED_SMALL_CENTROIDS_TSV = ROOT / "results/annotations/well_centroids_fixed_small_mapped.tsv"
LIVE_SMALL_CENTROIDS_TSV = ROOT / "results/annotations/well_centroids_small_mapped.tsv"

QC_ROOT = ROOT / "results/qc/03_live_fixed_alignment"
FIXED_QC_DIR = QC_ROOT / "fixed_small_large_alignment"
FIXED_QC_SUMMARY_TSV = QC_ROOT / "fixed_small_large_alignment_summary.tsv"
LIVE_FIXED_WELL_PROCESSING_QC_DIR = QC_ROOT / "live_fixed_well_processing"
LIVE_FIXED_WELL_PROCESSING_QC_SUMMARY_TSV = QC_ROOT / "live_fixed_well_processing_summary.tsv"
LIVE_FIXED_WELL_QC_DIR = QC_ROOT / "live_fixed_well_detection"
LIVE_FIXED_WELL_QC_SUMMARY_TSV = QC_ROOT / "live_fixed_well_detection_summary.tsv"
LIVE_FIXED_QC_DIR = QC_ROOT / "live_fixed_alignment"
LIVE_FIXED_QC_SUMMARY_TSV = QC_ROOT / "live_fixed_alignment_summary.tsv"
LIVE_FIXED_DAPI_OVERLAY_QC_DIR = QC_ROOT / "live_fixed_dapi_overlay"
LIVE_FIXED_DAPI_OVERLAY_QC_SUMMARY_TSV = QC_ROOT / "live_fixed_dapi_overlay_summary.tsv"

for p in [QC_ROOT, FIXED_QC_DIR, LIVE_FIXED_WELL_PROCESSING_QC_DIR, LIVE_FIXED_WELL_QC_DIR, LIVE_FIXED_QC_DIR, LIVE_FIXED_DAPI_OVERLAY_QC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("LIVE_SMALL_LARGE_TSV:", LIVE_SMALL_LARGE_TSV)
print("FIXED_SMALL_LARGE_TSV:", FIXED_SMALL_LARGE_TSV)
print("LIVE_FIXED_TSV:", LIVE_FIXED_TSV)
print("FIXED_SMALL_CENTROIDS_TSV:", FIXED_SMALL_CENTROIDS_TSV)
print("LIVE_SMALL_CENTROIDS_TSV:", LIVE_SMALL_CENTROIDS_TSV)


## Fixed Small-Large Pair Discovery

In [ ]:
FORCE_RECOMPUTE_FIXED_SMALL_LARGE = False
FORCE_RECOMPUTE_LIVE_FIXED = False
# Scratch-safe default: regenerate QC when its summary table is missing.
RENDER_FIXED_SMALL_LARGE_QC = not FIXED_QC_SUMMARY_TSV.exists()
SHOW_FIXED_SMALL_LARGE_QC_PANELS = False
LIVE_FIXED_WELL_RADII_PX = tuple(range(50, 61, 2))
LIVE_FIXED_WELL_MIN_CENTER_DISTANCE_PX = 80
LIVE_FIXED_OUTSIDE_SMALL_FOV_ONLY_POSITIONS = ["2-1"]
LIVE_FIXED_FORCE_RELAXED_WELL_DETECTION_POSITIONS = ["2-1"]
LIVE_FIXED_SKIP_WELL_REFINEMENT_POSITIONS = []
INTERMEDIATE_QC_REPRESENTATIVE_POSITIONS = ["1-1", "2-1", "2-6", "5-6"]
RENDER_LIVE_FIXED_WELL_PROCESSING_QC = not LIVE_FIXED_WELL_PROCESSING_QC_SUMMARY_TSV.exists()
SHOW_LIVE_FIXED_WELL_PROCESSING_QC_PANELS = False
LIVE_FIXED_WELL_PROCESSING_DEBUG_POSITION = "5-6"
RENDER_LIVE_FIXED_WELL_QC = not LIVE_FIXED_WELL_QC_SUMMARY_TSV.exists()
SHOW_LIVE_FIXED_WELL_QC_PANELS = False
RENDER_LIVE_FIXED_QC = not LIVE_FIXED_QC_SUMMARY_TSV.exists()
SHOW_LIVE_FIXED_QC_PANELS = False
FIXED_SMALL_LARGE_FALLBACK_CHANNEL_KEYWORDS = ["dapi"]
FIXED_SMALL_LARGE_FALLBACK_MAX_CENTER_OFFSET_PX = 200.0

print("FORCE_RECOMPUTE_FIXED_SMALL_LARGE:", FORCE_RECOMPUTE_FIXED_SMALL_LARGE)
print("FORCE_RECOMPUTE_LIVE_FIXED:", FORCE_RECOMPUTE_LIVE_FIXED)
print("RENDER_FIXED_SMALL_LARGE_QC:", RENDER_FIXED_SMALL_LARGE_QC)
print("SHOW_FIXED_SMALL_LARGE_QC_PANELS:", SHOW_FIXED_SMALL_LARGE_QC_PANELS)
print("LIVE_FIXED_WELL_RADII_PX:", LIVE_FIXED_WELL_RADII_PX)
print("LIVE_FIXED_WELL_MIN_CENTER_DISTANCE_PX:", LIVE_FIXED_WELL_MIN_CENTER_DISTANCE_PX)
print("LIVE_FIXED_OUTSIDE_SMALL_FOV_ONLY_POSITIONS:", LIVE_FIXED_OUTSIDE_SMALL_FOV_ONLY_POSITIONS)
print("LIVE_FIXED_FORCE_RELAXED_WELL_DETECTION_POSITIONS:", LIVE_FIXED_FORCE_RELAXED_WELL_DETECTION_POSITIONS)
print("LIVE_FIXED_SKIP_WELL_REFINEMENT_POSITIONS:", LIVE_FIXED_SKIP_WELL_REFINEMENT_POSITIONS)
print("INTERMEDIATE_QC_REPRESENTATIVE_POSITIONS:", INTERMEDIATE_QC_REPRESENTATIVE_POSITIONS)
print("RENDER_LIVE_FIXED_WELL_PROCESSING_QC:", RENDER_LIVE_FIXED_WELL_PROCESSING_QC)
print("SHOW_LIVE_FIXED_WELL_PROCESSING_QC_PANELS:", SHOW_LIVE_FIXED_WELL_PROCESSING_QC_PANELS)
print("LIVE_FIXED_WELL_PROCESSING_DEBUG_POSITION:", LIVE_FIXED_WELL_PROCESSING_DEBUG_POSITION)
print("RENDER_LIVE_FIXED_WELL_QC:", RENDER_LIVE_FIXED_WELL_QC)
print("SHOW_LIVE_FIXED_WELL_QC_PANELS:", SHOW_LIVE_FIXED_WELL_QC_PANELS)
print("RENDER_LIVE_FIXED_QC:", RENDER_LIVE_FIXED_QC)
print("SHOW_LIVE_FIXED_QC_PANELS:", SHOW_LIVE_FIXED_QC_PANELS)
print("FIXED_SMALL_LARGE_FALLBACK_CHANNEL_KEYWORDS:", FIXED_SMALL_LARGE_FALLBACK_CHANNEL_KEYWORDS)
print("FIXED_SMALL_LARGE_FALLBACK_MAX_CENTER_OFFSET_PX:", FIXED_SMALL_LARGE_FALLBACK_MAX_CENTER_OFFSET_PX)

FINAL_OVERLAY_REPRESENTATIVE_POSITIONS = ["1-1", "2-6", "5-6"]
print("FINAL_OVERLAY_REPRESENTATIVE_POSITIONS:", FINAL_OVERLAY_REPRESENTATIVE_POSITIONS)


In [ ]:
fixed_pairs = mwa.paired_records_from_directory(
    root=ROOT,
    target_subdir="data/2026-01-22_PDMS/day2-fix-well2-568SOX2-647T/single",
    cohort_id="2026-01-22_day2_fix",
)
print("Fixed pair count:", len(fixed_pairs))
display(pd.DataFrame([p.__dict__ for p in fixed_pairs]).head())

## Compute Fixed Small-Large Transforms

In [ ]:
fixed_transform_df = mwa.compute_pair_transforms(
    pairs=fixed_pairs,
    root=ROOT,
    transform_path=FIXED_SMALL_LARGE_TSV,
    force_recompute=FORCE_RECOMPUTE_FIXED_SMALL_LARGE,
    fallback_registration_channel_keywords=FIXED_SMALL_LARGE_FALLBACK_CHANNEL_KEYWORDS,
    fallback_max_center_offset_px=FIXED_SMALL_LARGE_FALLBACK_MAX_CENTER_OFFSET_PX,
    verbose=True,
)
print()
print("Fixed registration status counts:")
print(fixed_transform_df["registration_status"].value_counts(dropna=False).to_string())
display(fixed_transform_df.head())


## Render Fixed Small-Large QC

In [ ]:
if RENDER_FIXED_SMALL_LARGE_QC:
    fixed_qc_df = mwa.render_alignment_qc_for_pairs(
        pairs=fixed_pairs,
        root=ROOT,
        transform_path=FIXED_SMALL_LARGE_TSV,
        qc_dir=FIXED_QC_DIR,
        show_inline=False,
        verbose=True,
    )
    fixed_qc_df.to_csv(FIXED_QC_SUMMARY_TSV, sep="	", index=False)
    print("Saved:", FIXED_QC_SUMMARY_TSV)
    print(fixed_qc_df["qc_status"].value_counts(dropna=False).to_string())
    display(fixed_qc_df.head())
else:
    print("Skipping fixed small-large QC rendering. Set RENDER_FIXED_SMALL_LARGE_QC=True to refresh PNGs.")
    if FIXED_QC_SUMMARY_TSV.exists():
        fixed_qc_df = pd.read_csv(FIXED_QC_SUMMARY_TSV, sep="	")
        print("Loaded existing summary:", FIXED_QC_SUMMARY_TSV)
        display(fixed_qc_df.head())
    else:
        fixed_qc_df = pd.DataFrame()
        print("No existing fixed small-large QC summary found.")


## Live-Fixed Pair Discovery

In [ ]:
live_tr_df = mwa.load_pair_transform_table(LIVE_SMALL_LARGE_TSV)
fixed_tr_df = mwa.load_pair_transform_table(FIXED_SMALL_LARGE_TSV)
live_by_pos = {str(r.canonical_position): pd.Series(r._asdict()) for r in live_tr_df.itertuples(index=False)}
fixed_by_pos = {str(r.canonical_position): pd.Series(r._asdict()) for r in fixed_tr_df.itertuples(index=False)}
common_positions = sorted(set(live_by_pos) & set(fixed_by_pos))

live_fixed_records = []
for pos in common_positions:
    live_row = live_by_pos[pos]
    fixed_row = fixed_by_pos[pos]
    live_pair = mwa.PairedImageRecord(
        image_id=str(live_row["image_id"]),
        cohort_id=str(live_row["cohort_id"]),
        canonical_position=str(pos),
        small_file_path=str(live_row["small_file_path"]),
        large_file_path=str(live_row["large_file_path"]),
    )
    fixed_pair = mwa.PairedImageRecord(
        image_id=str(fixed_row["image_id"]),
        cohort_id=str(fixed_row["cohort_id"]),
        canonical_position=str(pos),
        small_file_path=str(fixed_row["small_file_path"]),
        large_file_path=str(fixed_row["large_file_path"]),
    )
    live_fixed_records.append(
        lfa.LiveFixedPairRecord(
            canonical_position=str(pos),
            live_pair=live_pair,
            fixed_pair=fixed_pair,
        )
    )

need_live_fixed_recompute = bool(FORCE_RECOMPUTE_LIVE_FIXED) or (not LIVE_FIXED_TSV.exists())
if LIVE_FIXED_TSV.exists() and not need_live_fixed_recompute:
    existing_live_fixed_df = pd.read_csv(LIVE_FIXED_TSV, sep="	")
    existing_live_fixed_df["canonical_position"] = existing_live_fixed_df["canonical_position"].astype(str)
    skip_positions = {str(p) for p in LIVE_FIXED_SKIP_WELL_REFINEMENT_POSITIONS}
    if skip_positions:
        skip_rows = existing_live_fixed_df[existing_live_fixed_df["canonical_position"].isin(skip_positions)].copy()
        if len(skip_rows) < len(skip_positions) or not skip_rows["well_refine_status"].astype(str).eq("skipped_by_position").all():
            need_live_fixed_recompute = True
            print("Existing live-fixed alignment table does not reflect requested skip-refine positions; recomputing.")

    upstream_paths = [LIVE_SMALL_LARGE_TSV, FIXED_SMALL_LARGE_TSV, ROOT / "scripts/live_fixed_alignment.py"]
    upstream_latest_mtime = max(path.stat().st_mtime for path in upstream_paths if path.exists())
    live_fixed_mtime = LIVE_FIXED_TSV.stat().st_mtime
    if upstream_latest_mtime > live_fixed_mtime:
        need_live_fixed_recompute = True
        print("Upstream small-large transform table is newer than live-fixed alignment table; recomputing.")

print("Live-fixed record count:", len(live_fixed_records))
print("need_live_fixed_recompute:", need_live_fixed_recompute)
display(pd.DataFrame([
    {
        "canonical_position": r.canonical_position,
        "live_small": r.live_pair.small_file_path,
        "fixed_small": r.fixed_pair.small_file_path,
        "live_large": r.live_pair.large_file_path,
        "fixed_large": r.fixed_pair.large_file_path,
    }
    for r in live_fixed_records
]).head())


## Compute Fixed-to-Live Alignment Table

In [ ]:
if need_live_fixed_recompute:
    live_fixed_df = lfa.compute_fixed_to_live_alignment_table(
        records=live_fixed_records,
        root=ROOT,
        live_transform_path=LIVE_SMALL_LARGE_TSV,
        fixed_transform_path=FIXED_SMALL_LARGE_TSV,
        out_path=LIVE_FIXED_TSV,
        force_recompute=True,
        well_detect_radii_px=LIVE_FIXED_WELL_RADII_PX,
        well_detect_min_center_distance_px=LIVE_FIXED_WELL_MIN_CENTER_DISTANCE_PX,
        well_refine_outside_small_fov_only_positions=LIVE_FIXED_OUTSIDE_SMALL_FOV_ONLY_POSITIONS,
        well_refine_force_relaxed_detection_positions=LIVE_FIXED_FORCE_RELAXED_WELL_DETECTION_POSITIONS,
        well_refine_skip_positions=LIVE_FIXED_SKIP_WELL_REFINEMENT_POSITIONS,
        verbose=True,
    )
else:
    live_fixed_df = pd.read_csv(LIVE_FIXED_TSV, sep="	")
    print("Loaded existing live-fixed alignment table:", LIVE_FIXED_TSV)

print()
print("Live-fixed status counts:")
print(live_fixed_df["status"].value_counts(dropna=False).to_string())
display(live_fixed_df.head())

## Build Fixed Small Mapped Bead Table
This table belongs in the alignment layer because it combines live large bead annotations with the fixed/live alignment and fixed small-large transforms.

In [ ]:
fixed_small_centroids_df = lfq.build_fixed_small_mapped_centroid_table(
    large_centroid_path=ROOT / "results/annotations/well_centroids_large.tsv",
    live_fixed_alignment_path=LIVE_FIXED_TSV,
    fixed_transform_path=FIXED_SMALL_LARGE_TSV,
    out_path=FIXED_SMALL_CENTROIDS_TSV,
    verbose=True,
)
print("Mapping status counts:")
print(fixed_small_centroids_df["mapping_status"].value_counts(dropna=False).to_string())
display(fixed_small_centroids_df.head())


## Live-Fixed Alignment Summary

In [ ]:
ok_live_fixed = live_fixed_df[live_fixed_df["status"] == "ok"].copy()
fig, axes = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)
axes[0].hist(ok_live_fixed["fixed_large_to_live_large_theta_deg"].astype(float), bins=16, color="tab:blue")
axes[0].set_title("Coarse rigid theta (deg)")
axes[0].set_xlabel("theta_deg")
axes[0].set_ylabel("count")
axes[1].scatter(
    ok_live_fixed["fixed_large_to_live_large_shift_x_px"].astype(float),
    ok_live_fixed["fixed_large_to_live_large_shift_y_px"].astype(float),
    s=18,
    color="tab:green",
)
axes[1].axhline(0.0, color="0.7", lw=1)
axes[1].axvline(0.0, color="0.7", lw=1)
axes[1].set_title("Coarse rigid translation (px)")
axes[1].set_xlabel("shift_x_px")
axes[1].set_ylabel("shift_y_px")
axes[2].hist(ok_live_fixed["fixed_large_to_live_large_score"].astype(float), bins=16, color="tab:purple")
axes[2].set_title("Coarse alignment score")
axes[2].set_xlabel("score")
refine_counts = ok_live_fixed["well_refine_status"].fillna("missing").astype(str).value_counts()
axes[3].bar(refine_counts.index.astype(str), refine_counts.values.astype(int), color="tab:orange")
axes[3].set_title("Large-well refinement status")
axes[3].set_ylabel("count")
axes[3].tick_params(axis="x", rotation=25)
plt.show()

print("Well refinement pair counts:")
print(ok_live_fixed["well_refine_n_pairs"].fillna(0).astype(int).value_counts().sort_index().to_string())
print("\nLarge-well candidate counts (live):")
print(ok_live_fixed["well_refine_n_live_candidates"].fillna(0).astype(int).value_counts().sort_index().to_string())
print("\nLarge-well candidate counts (fixed):")
print(ok_live_fixed["well_refine_n_fixed_candidates"].fillna(0).astype(int).value_counts().sort_index().to_string())

display(
    ok_live_fixed[[
        "canonical_position",
        "fixed_large_to_live_large_score",
        "well_refine_status",
        "well_refine_n_pairs",
        "well_refine_n_live_candidates",
        "well_refine_n_fixed_candidates",
        "well_refine_delta_theta_deg",
        "well_refine_delta_shift_x_px",
        "well_refine_delta_shift_y_px",
        "well_refine_mean_residual_px",
        "well_refine_max_residual_px",
    ]].head(12)
)


## Render Live-Fixed Well Processing Debug

In [ ]:
debug_positions = None if SHOW_LIVE_FIXED_WELL_PROCESSING_QC_PANELS else INTERMEDIATE_QC_REPRESENTATIVE_POSITIONS

if RENDER_LIVE_FIXED_WELL_PROCESSING_QC:
    live_fixed_well_processing_qc_df = lfa.render_live_fixed_large_well_processing_qc_batch(
        records=live_fixed_records,
        align_path=LIVE_FIXED_TSV,
        live_transform_path=LIVE_SMALL_LARGE_TSV,
        fixed_transform_path=FIXED_SMALL_LARGE_TSV,
        root=ROOT,
        out_dir=LIVE_FIXED_WELL_PROCESSING_QC_DIR,
        positions=debug_positions,
        well_detect_radii_px=LIVE_FIXED_WELL_RADII_PX,
        well_detect_min_center_distance_px=LIVE_FIXED_WELL_MIN_CENTER_DISTANCE_PX,
        show_inline=False,
        verbose=True,
    )
    live_fixed_well_processing_qc_df.to_csv(LIVE_FIXED_WELL_PROCESSING_QC_SUMMARY_TSV, sep="\t", index=False)
    print("Saved:", LIVE_FIXED_WELL_PROCESSING_QC_SUMMARY_TSV)
    print("Processing debug positions:", "ALL" if SHOW_LIVE_FIXED_WELL_PROCESSING_QC_PANELS else INTERMEDIATE_QC_REPRESENTATIVE_POSITIONS)
    print(live_fixed_well_processing_qc_df["qc_status"].value_counts(dropna=False).to_string())
    display(live_fixed_well_processing_qc_df.head())
else:
    print("Skipping live-fixed well-processing debug rendering. Set RENDER_LIVE_FIXED_WELL_PROCESSING_QC=True to refresh PNGs.")
    if LIVE_FIXED_WELL_PROCESSING_QC_SUMMARY_TSV.exists():
        live_fixed_well_processing_qc_df = pd.read_csv(LIVE_FIXED_WELL_PROCESSING_QC_SUMMARY_TSV, sep="\t")
        print("Loaded existing summary:", LIVE_FIXED_WELL_PROCESSING_QC_SUMMARY_TSV)
        display(live_fixed_well_processing_qc_df.head())
    else:
        live_fixed_well_processing_qc_df = pd.DataFrame()
        print("No existing live-fixed well-processing QC summary found.")


## Optional Live-Fixed Well Processing Debug Panels

In [ ]:
if SHOW_LIVE_FIXED_WELL_PROCESSING_QC_PANELS:
    live_fixed_well_processing_qc_show_df = pd.read_csv(LIVE_FIXED_WELL_PROCESSING_QC_SUMMARY_TSV, sep="	")
    for _, rr in live_fixed_well_processing_qc_show_df.iterrows():
        if rr["qc_status"] != "rendered":
            continue
        print(rr["canonical_position"])
        display(Image(filename=str(rr["qc_png_path"])))
else:
    print("Showing representative live-fixed processing-debug positions:", INTERMEDIATE_QC_REPRESENTATIVE_POSITIONS)
    rep_qc_df = pd.read_csv(LIVE_FIXED_WELL_PROCESSING_QC_SUMMARY_TSV, sep="	")
    rep_qc_df = rep_qc_df[rep_qc_df["canonical_position"].astype(str).isin(INTERMEDIATE_QC_REPRESENTATIVE_POSITIONS)]
    for _, rr in rep_qc_df.iterrows():
        if rr["qc_status"] != "rendered":
            continue
        print(rr["canonical_position"])
        display(Image(filename=str(rr["qc_png_path"])))


## Render Live-Fixed Large Well Detection QC

In [ ]:
if RENDER_LIVE_FIXED_WELL_QC:
    live_fixed_well_qc_df = lfa.render_live_fixed_large_well_detection_qc_batch(
        records=live_fixed_records,
        align_path=LIVE_FIXED_TSV,
        live_transform_path=LIVE_SMALL_LARGE_TSV,
        fixed_transform_path=FIXED_SMALL_LARGE_TSV,
        root=ROOT,
        out_dir=LIVE_FIXED_WELL_QC_DIR,
        well_detect_radii_px=LIVE_FIXED_WELL_RADII_PX,
        well_detect_min_center_distance_px=LIVE_FIXED_WELL_MIN_CENTER_DISTANCE_PX,
        show_inline=False,
        verbose=True,
    )
    live_fixed_well_qc_df.to_csv(LIVE_FIXED_WELL_QC_SUMMARY_TSV, sep="\t", index=False)
    print("Saved:", LIVE_FIXED_WELL_QC_SUMMARY_TSV)
    print(live_fixed_well_qc_df["qc_status"].value_counts(dropna=False).to_string())
    display(live_fixed_well_qc_df.head())
else:
    print("Skipping live-fixed large well-detection QC rendering. Set RENDER_LIVE_FIXED_WELL_QC=True to refresh PNGs.")
    if LIVE_FIXED_WELL_QC_SUMMARY_TSV.exists():
        live_fixed_well_qc_df = pd.read_csv(LIVE_FIXED_WELL_QC_SUMMARY_TSV, sep="\t")
        print("Loaded existing summary:", LIVE_FIXED_WELL_QC_SUMMARY_TSV)
        display(live_fixed_well_qc_df.head())
    else:
        live_fixed_well_qc_df = pd.DataFrame()
        print("No existing live-fixed large well-detection QC summary found.")


## Optional Live-Fixed Large Well Detection QC Panels

In [ ]:
if SHOW_LIVE_FIXED_WELL_QC_PANELS:
    live_fixed_well_qc_show_df = pd.read_csv(LIVE_FIXED_WELL_QC_SUMMARY_TSV, sep="	")
    for _, rr in live_fixed_well_qc_show_df.iterrows():
        if rr["qc_status"] != "rendered":
            continue
        print(rr["canonical_position"])
        display(Image(filename=str(rr["qc_png_path"])))
else:
    print("Showing representative live-fixed large well-detection positions:", INTERMEDIATE_QC_REPRESENTATIVE_POSITIONS)
    rep_qc_df = pd.read_csv(LIVE_FIXED_WELL_QC_SUMMARY_TSV, sep="	")
    rep_qc_df = rep_qc_df[rep_qc_df["canonical_position"].astype(str).isin(INTERMEDIATE_QC_REPRESENTATIVE_POSITIONS)]
    for _, rr in rep_qc_df.iterrows():
        if rr["qc_status"] != "rendered":
            continue
        print(rr["canonical_position"])
        display(Image(filename=str(rr["qc_png_path"])))


## Render Live-Fixed QC

In [ ]:
if RENDER_LIVE_FIXED_QC:
    live_fixed_qc_df = lfa.render_live_fixed_alignment_qc_batch(
        records=live_fixed_records,
        align_path=LIVE_FIXED_TSV,
        root=ROOT,
        out_dir=LIVE_FIXED_QC_DIR,
        show_inline=False,
        verbose=True,
    )
    live_fixed_qc_df.to_csv(LIVE_FIXED_QC_SUMMARY_TSV, sep="\t", index=False)
    print("Saved:", LIVE_FIXED_QC_SUMMARY_TSV)
    print(live_fixed_qc_df["qc_status"].value_counts(dropna=False).to_string())
    display(live_fixed_qc_df.head())
else:
    print("Skipping live-fixed QC rendering. Set RENDER_LIVE_FIXED_QC=True to refresh PNGs.")
    if LIVE_FIXED_QC_SUMMARY_TSV.exists():
        live_fixed_qc_df = pd.read_csv(LIVE_FIXED_QC_SUMMARY_TSV, sep="\t")
        print("Loaded existing summary:", LIVE_FIXED_QC_SUMMARY_TSV)
        display(live_fixed_qc_df.head())
    else:
        live_fixed_qc_df = pd.DataFrame()
        print("No existing live-fixed QC summary found.")


## Fixed Small-Large QC Panels

Optional. Set `SHOW_FIXED_SMALL_LARGE_QC_PANELS=True` if you want to inspect the saved fixed small-large QC PNGs inline.

In [ ]:
if SHOW_FIXED_SMALL_LARGE_QC_PANELS:
    fixed_qc_show_df = pd.read_csv(FIXED_QC_SUMMARY_TSV, sep="	")
    for _, rr in fixed_qc_show_df.iterrows():
        if rr["qc_status"] != "rendered":
            continue
        print(rr["canonical_position"])
        display(Image(filename=str(ROOT / rr["saved_qc_path"])))
else:
    print("Showing representative fixed small-large positions:", INTERMEDIATE_QC_REPRESENTATIVE_POSITIONS)
    rep_qc_df = pd.read_csv(FIXED_QC_SUMMARY_TSV, sep="	")
    rep_qc_df = rep_qc_df[rep_qc_df["canonical_position"].astype(str).isin(INTERMEDIATE_QC_REPRESENTATIVE_POSITIONS)]
    for _, rr in rep_qc_df.iterrows():
        if rr["qc_status"] != "rendered":
            continue
        print(rr["canonical_position"])
        display(Image(filename=str(ROOT / rr["saved_qc_path"])))


## Optional Live-Fixed QC Panels

In [ ]:
if SHOW_LIVE_FIXED_QC_PANELS:
    live_fixed_qc_show_df = pd.read_csv(LIVE_FIXED_QC_SUMMARY_TSV, sep="	")
    for _, rr in live_fixed_qc_show_df.iterrows():
        if rr["qc_status"] != "rendered":
            continue
        print(rr["canonical_position"])
        display(Image(filename=str(rr["qc_png_path"])))
else:
    print("Showing representative live-fixed QC positions:", INTERMEDIATE_QC_REPRESENTATIVE_POSITIONS)
    rep_qc_df = pd.read_csv(LIVE_FIXED_QC_SUMMARY_TSV, sep="	")
    rep_qc_df = rep_qc_df[rep_qc_df["canonical_position"].astype(str).isin(INTERMEDIATE_QC_REPRESENTATIVE_POSITIONS)]
    for _, rr in rep_qc_df.iterrows():
        if rr["qc_status"] != "rendered":
            continue
        print(rr["canonical_position"])
        display(Image(filename=str(rr["qc_png_path"])))

## Follow-Up | Fixed DAPI Over Live BF

This is a simplified small-image overlay for manual alignment review: **warped fixed small DAPI** over **live small brightfield**, with **magenta circles marking bead-containing wells inside the live small-image FOV**, one scene at a time, for **all positions**.

In [ ]:
def robust_rescale_from_values(image: np.ndarray, valid_mask: np.ndarray | None = None, q_low: float = 1.0, q_high: float = 99.0) -> np.ndarray:
    arr = np.asarray(image, dtype=np.float32)
    if valid_mask is None:
        keep = np.isfinite(arr)
    else:
        keep = np.asarray(valid_mask, dtype=bool) & np.isfinite(arr)
    if not np.any(keep):
        return np.zeros_like(arr, dtype=np.float32)
    vals = arr[keep]
    lo = float(np.percentile(vals, q_low))
    hi = float(np.percentile(vals, q_high))
    if not np.isfinite(lo):
        lo = float(np.nanmin(vals))
    if not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0
    out = (arr - lo) / max(hi - lo, float(lfq.EPS))
    out = np.clip(out, 0.0, 1.0)
    out[~np.isfinite(out)] = 0.0
    return out.astype(np.float32)


def load_live_small_bf(path_like):
    img = lfq.common.read_czi(ROOT / str(path_like))
    bf_idx = lfq.common.find_channel_index(img.channels, ["bright"])
    if bf_idx is None:
        raise RuntimeError(f"Missing live BF channel in {path_like}; channels={img.channels}")
    return img.channel_images[int(bf_idx)].astype(np.float32)


def load_fixed_small_dapi_max(path_like):
    bundle = lfq.load_czi_bundle_with_z(ROOT / str(path_like))
    dapi_idx = lfq.common.find_channel_index(bundle["channels"], ["dapi"])
    if dapi_idx is None:
        raise RuntimeError(f"Missing fixed DAPI channel in {path_like}; channels={bundle['channels']}")
    return np.asarray(bundle["channel_max_cyx"][int(dapi_idx)], dtype=np.float32)


live_cent_df = pd.read_csv(LIVE_SMALL_CENTROIDS_TSV, sep="\t")
live_cent_df["canonical_position"] = live_cent_df["canonical_position"].astype(str)

overlay_rows = []
ok_df = live_fixed_df[live_fixed_df["status"].astype(str) == "ok"].copy()
ok_df["canonical_position"] = ok_df["canonical_position"].astype(str)
show_positions = [str(p) for p in FINAL_OVERLAY_REPRESENTATIVE_POSITIONS]
ok_df = ok_df[ok_df["canonical_position"].isin(show_positions)].copy()

for _, rr in ok_df.sort_values("canonical_position").iterrows():
    pos = str(rr["canonical_position"])
    try:
        live_bf = load_live_small_bf(rr["live_small_file_path"])
        fixed_dapi = load_fixed_small_dapi_max(rr["fixed_small_file_path"])
        fixed_to_live = lfa.affine_matrix_from_row_prefix(rr, "fixed_small_to_live_small")
        warped_fixed_dapi = lfa.warp_image_with_affine(
            image=fixed_dapi,
            forward_mat=fixed_to_live,
            output_shape_yx=live_bf.shape,
            order=1,
            cval=np.nan,
        )

        cent_sub = live_cent_df[(live_cent_df["canonical_position"].astype(str) == pos) & (live_cent_df["mapping_status"] == "ok") & (live_cent_df["annotation_status"] == "annotated") & (live_cent_df["inside_small_fov"].fillna(False).astype(bool))].copy()
        beads_xy = cent_sub[["centroid_small_x_px", "centroid_small_y_px"]].to_numpy(dtype=np.float32) if len(cent_sub) else np.zeros((0, 2), dtype=np.float32)

        fig, ax = plt.subplots(1, 1, figsize=(8.8, 8.8), constrained_layout=True)
        ax.imshow(robust_rescale_from_values(live_bf), cmap="gray")
        ax.imshow(
            robust_rescale_from_values(warped_fixed_dapi, np.isfinite(warped_fixed_dapi)),
            cmap="magma",
            alpha=0.58,
        )
        if len(beads_xy):
            ax.scatter(
                beads_xy[:, 0],
                beads_xy[:, 1],
                s=180,
                facecolors="none",
                edgecolors="magenta",
                linewidths=2.2,
            )

        ax.set_title(
            f"{pos} | warped fixed DAPI over live BF\n"
            f"alignment score={float(rr['fixed_large_to_live_large_score']):.4f} | magenta=bead-containing wells in small FOV"
        )
        ax.axis("off")
        out_png = LIVE_FIXED_DAPI_OVERLAY_QC_DIR / f"{pos}_live_fixed_dapi_overlay_qc.png"
        fig.savefig(out_png, dpi=160, bbox_inches="tight")
        plt.close(fig)
        overlay_rows.append({
            "canonical_position": pos,
            "qc_status": "rendered",
            "qc_png_path": str(out_png),
        })
    except Exception as exc:
        overlay_rows.append({
            "canonical_position": pos,
            "qc_status": f"error:{exc}",
            "qc_png_path": np.nan,
        })

live_fixed_dapi_overlay_qc_df = pd.DataFrame(overlay_rows).sort_values("canonical_position").reset_index(drop=True)
live_fixed_dapi_overlay_qc_df.to_csv(LIVE_FIXED_DAPI_OVERLAY_QC_SUMMARY_TSV, sep="	", index=False)
print("Showing representative overlay positions:", show_positions)
print(live_fixed_dapi_overlay_qc_df["qc_status"].value_counts(dropna=False).to_string())
display(live_fixed_dapi_overlay_qc_df)

for _, row in live_fixed_dapi_overlay_qc_df.iterrows():
    qc_path = row.get("qc_png_path")
    if isinstance(qc_path, str) and qc_path:
        display(Image(filename=qc_path, width=980))